# Практика 11. Визуализация данных в Python

На прошлой паре мы посчитали, что средний рейс стоит 44 262 руб., а медианный — 38 125,
и объясняли словами, почему они различаются. Сегодня то же самое станет **видно с первого
взгляда**: одна гистограмма заменяет абзац объяснений.

**Данные те же:** 910 рейсов за февраль–май 2025.

| Часть | О чём |
|---|---|
| 1 | Как устроен график в matplotlib |
| 2 | Выбор типа визуализации под задачу |
| 3 | Оформление: подписи, легенды, масштабы, аннотации |
| 4 | seaborn: меньше кода, больше смысла |
| 5 | Plotly: интерактивные графики |
| 6 | Мини-проект: дашборд из четырёх графиков |

> **Главная мысль пары:** график — не украшение отчёта, а **инструмент доказательства**.
> У него одна задача: чтобы читатель увидел то, что вы хотите сказать, за три секунды
> и без ошибок. Поэтому половина сегодняшней работы — не рисование, а оформление:
> подписи, масштаб, порядок категорий.

---
## Шаг 0. Проект

```
# терминал
mkdir data-viz
cd data-viz
```

`requirements.txt`:

```
pandas
matplotlib
seaborn
plotly
```

```
# терминал (Windows)
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
```

```
# терминал (macOS / Linux)
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

| Библиотека | Зачем |
|---|---|
| `matplotlib` | основа: на ней построено всё остальное, даёт полный контроль |
| `seaborn` | надстройка над matplotlib: красивее и короче для типовых графиков |
| `plotly` | интерактивные графики: наведение мышкой, зум, html-файл |

---
## Данные

In [ ]:
# ЯЧЕЙКА-ГЕНЕРАТОР ДАННЫХ. Запустите её (Shift+Enter), менять ничего не нужно.
# Создаёт data/trips_q.csv — чистый журнал рейсов за февраль-май 2025.

import csv
import random
from pathlib import Path

random.seed(10)

routes = {
    "Москва-Казань": 820, "Москва-Санкт-Петербург": 710, "Москва-Нижний Новгород": 420,
    "Москва-Воронеж": 520, "Казань-Самара": 360, "Екатеринбург-Челябинск": 220,
    "Новосибирск-Омск": 680, "Санкт-Петербург-Петрозаводск": 430,
}
carriers = ["ТрансЛогистик", "РусКарго", "СибПуть", "ВолгаТранс", "АвтоДорСервис"]
tariffs = {"Контейнер": 7.5, "Нефтепродукты": 9.2, "Зерно": 5.1,
           "Металл": 6.8, "Лес": 4.6, "Стройматериалы": 5.9}
clients = ["Агрохолдинг Нива", "МеталлПром", "СтройБаза 77",
           "НефтеТрейд", "ЛесЭкспорт", "РитейлСеть"]
days_in_month = {2: 28, 3: 31, 4: 30, 5: 31}

rows = []
trip_no = 1
for month in (2, 3, 4, 5):
    # к лету перевозок становится больше — будет видно на графике динамики
    n_trips = {2: 190, 3: 215, 4: 240, 5: 265}[month]
    for _ in range(n_trips):
        route, distance = random.choice(list(routes.items()))
        cargo = random.choice(list(tariffs))
        carrier = random.choice(carriers)
        weight = round(min(22.0, max(2.0, random.gauss(13, 4.5))), 1)

        # задержки: у большинства рейсов их нет, у остальных — длинный хвост
        if random.random() < 0.72:
            delay = 0
        else:
            delay = random.choice([1, 1, 2, 2, 3, 4, 6, 8, 12, 18])

        speed = random.uniform(46, 64)
        duration = round(distance / speed + delay, 1)
        cost = distance * weight * tariffs[cargo] * random.uniform(0.94, 1.09)

        rows.append({
            "trip_id": f"T{trip_no:04d}",
            "date": f"2025-{month:02d}-{random.randint(1, days_in_month[month]):02d}",
            "month": f"2025-{month:02d}",
            "route": route,
            "carrier": carrier,
            "cargo_type": cargo,
            "client": random.choice(clients),
            "weight_t": weight,
            "distance_km": distance,
            "duration_h": duration,
            "cost_rub": int(round(cost, -1)),
            "delay_h": delay,
        })
        trip_no += 1

Path("data").mkdir(exist_ok=True)
with open("data/trips_q.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Готово. Создан файл data/trips_q.csv")
print("Рейсов:", len(rows))
print("Период: февраль - май 2025")
print("Колонки:", list(rows[0].keys()))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/trips_q.csv", parse_dates=["date"])

# пара производных показателей — понадобятся для графиков
df["cost_per_km"] = (df["cost_rub"] / df["distance_km"]).round(2)
df["on_time"] = df["delay_h"] == 0

# единый стиль для всех графиков в ноутбуке
sns.set_theme(style="whitegrid", font_scale=1.0)

print("рейсов:", len(df))
df.head(3)

---
# Часть 1. Как устроен график

В matplotlib у графика два уровня:

- **`Figure`** — лист бумаги, на котором рисуем;
- **`Axes`** — система координат с осями (на одном листе их может быть несколько).

Создают их вместе:

```python
fig, ax = plt.subplots(figsize=(8, 4))   # лист 8x4 дюйма и одна система координат
```

Дальше всё рисование идёт через `ax`: `ax.bar(...)`, `ax.set_title(...)`, `ax.set_xlabel(...)`.

> Вы встретите в интернете другой стиль — `plt.bar(...)`, `plt.title(...)` без `ax`.
> Он работает, но рисует «в текущий график», и как только графиков становится несколько,
> начинается путаница. **Привыкайте сразу к `fig, ax`** — этот стиль работает всегда.

In [ ]:
# 1. Готовим данные: выручка по маршрутам, по возрастанию
revenue_by_route = df.groupby("route")["cost_rub"].sum().sort_values()

# 2. Создаём лист и систему координат
fig, ax = plt.subplots(figsize=(9, 4.5))

# 3. Рисуем горизонтальные столбцы
ax.barh(revenue_by_route.index, revenue_by_route.values / 1_000_000, color="#4C72B0")

# 4. Подписываем — без этого график бесполезен
ax.set_title("Выручка по маршрутам, февраль-май 2025")
ax.set_xlabel("Выручка, млн руб.")

# 5. Показываем
plt.show()

Разберём два решения, принятые в этом коде, — они важнее самого рисования.

**Почему `barh`, а не `bar`.** Названия маршрутов длинные. На вертикальных столбцах они
налезают друг на друга и их приходится поворачивать, а повёрнутый текст читается плохо.
Правило: **длинные подписи категорий — горизонтальные столбцы**.

**Почему данные отсортированы.** Отсортированный график отвечает на вопрос «кто больше»
мгновенно, несортированный заставляет сравнивать столбцы глазами. Правило:
**категории на столбчатой диаграмме сортируют по значению** — кроме случая, когда у них
есть собственный порядок (месяцы, дни недели, размеры S/M/L).

Деление на миллион — тоже решение: подпись `7692450` читать неудобно, `7.7` — удобно.
Единицы при этом вынесены в название оси.

---
# Часть 2. Выбор типа визуализации

Тип графика выбирается не по вкусу, а по вопросу, на который он отвечает.

| Вопрос | Тип графика | Чем рисуем |
|---|---|---|
| Сравнить категории между собой | столбчатая | `ax.bar` / `ax.barh` |
| Как показатель менялся во времени | линейная | `ax.plot` |
| Как распределены значения | гистограмма | `ax.hist` |
| Есть ли связь двух величин | точечная | `ax.scatter` |
| Сравнить распределения по группам | «ящик с усами» | `sns.boxplot` |
| Значения в двух измерениях сразу | теплокарта | `sns.heatmap` |

Четыре основных типа на наших данных — в одной сетке. Обратите внимание на устройство
`plt.subplots(2, 2)`: он возвращает массив систем координат, к каждой обращаемся по индексу.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. Сравнение категорий: выручка по типам груза
cargo = df.groupby("cargo_type")["cost_rub"].sum().sort_values()
axes[0, 0].barh(cargo.index, cargo.values / 1_000_000, color="#4C72B0")
axes[0, 0].set_title("Сравнение: выручка по типам груза")
axes[0, 0].set_xlabel("млн руб.")

# 2. Динамика: выручка по месяцам
monthly = df.groupby("month")["cost_rub"].sum() / 1_000_000
axes[0, 1].plot(monthly.index, monthly.values, marker="o", color="#C44E52")
axes[0, 1].set_title("Динамика: выручка по месяцам")
axes[0, 1].set_ylabel("млн руб.")
axes[0, 1].set_ylim(0, monthly.max() * 1.15)          # ось от нуля — см. часть 3

# 3. Распределение: стоимость рейса
axes[1, 0].hist(df["cost_rub"] / 1000, bins=30, color="#55A868", edgecolor="white")
axes[1, 0].set_title("Распределение: стоимость рейса")
axes[1, 0].set_xlabel("тыс. руб.")
axes[1, 0].set_ylabel("число рейсов")

# 4. Связь: вес и стоимость
axes[1, 1].scatter(df["weight_t"], df["cost_rub"] / 1000, alpha=0.3, s=15, color="#8172B2")
axes[1, 1].set_title("Связь: вес и стоимость рейса")
axes[1, 1].set_xlabel("вес, т")
axes[1, 1].set_ylabel("стоимость, тыс. руб.")

fig.tight_layout()
plt.show()

Два приёма из этого кода, которые стоит забрать:

- **`fig.tight_layout()`** — автоматически расставляет отступы, чтобы подписи не налезали
  друг на друга. Вызывайте всегда перед `plt.show()`;
- **`alpha=0.3`** на точечном графике — полупрозрачность. Когда точек много, они
  накладываются, и без прозрачности видно только чёрное пятно. Прозрачность показывает,
  где точек больше.

## Гистограмма: то, что мы считали на прошлой паре

Гистограмма делит диапазон на интервалы и показывает, сколько значений попало в каждый.
Именно она делает перекос из темы 10 видимым: добавим на неё среднее и медиану.

In [ ]:
mean_cost = df["cost_rub"].mean()
median_cost = df["cost_rub"].median()

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.hist(df["cost_rub"] / 1000, bins=35, color="#55A868", edgecolor="white")

# Вертикальные линии: среднее и медиана
ax.axvline(median_cost / 1000, color="#C44E52", linestyle="--", linewidth=2,
           label=f"медиана: {median_cost/1000:.1f} тыс.")
ax.axvline(mean_cost / 1000, color="#4C72B0", linestyle="-", linewidth=2,
           label=f"среднее: {mean_cost/1000:.1f} тыс.")

ax.set_title("Распределение стоимости рейса: среднее правее медианы")
ax.set_xlabel("Стоимость рейса, тыс. руб.")
ax.set_ylabel("Число рейсов")
ax.legend()

fig.tight_layout()
plt.show()

Теперь видно глазами то, что на прошлой паре мы объясняли словами: у распределения
**длинный правый хвост** — немного очень дорогих рейсов. Хвост тянет среднее вправо,
медиана остаётся на месте. Отсюда и 42% рейсов дороже среднего вместо ожидаемой половины.

Это и есть сила визуализации: один график объясняет то, на что уходит абзац текста.

## Чего делать не надо

| Приём | Почему плохо | Что вместо |
|---|---|---|
| круговая диаграмма на 6+ категориях | глаз не сравнивает углы; подписи не влезают | `barh` с сортировкой |
| 3D-столбцы, тени, градиенты | украшения мешают читать значения | плоские столбцы |
| своя яркая палитра на каждый график | читатель заново разбирается в цветах | один стиль на весь отчёт |
| легенда при одной серии данных | занимает место, ничего не сообщает | название графика |

Сравните сами: два графика ниже построены по одним и тем же данным.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

cargo_sorted = df.groupby("cargo_type")["cost_rub"].sum().sort_values()

# Плохо: круговая диаграмма
axes[0].pie(cargo_sorted.values, labels=cargo_sorted.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("Плохо: доли на круговой диаграмме")

# Хорошо: столбцы с сортировкой
axes[1].barh(cargo_sorted.index, cargo_sorted.values / 1_000_000, color="#4C72B0")
axes[1].set_title("Хорошо: те же данные столбцами")
axes[1].set_xlabel("Выручка, млн руб.")

fig.tight_layout()
plt.show()

На круговой диаграмме попробуйте быстро ответить, что больше — зерно или лес.
На столбчатой ответ виден сразу, потому что длины сравнивать легко, а углы — нет.

### Задание 1

Постройте три графика, каждый — ответ на свой вопрос. Тип выбирайте по таблице выше.

1. **«Какой перевозчик привёз больше всего выручки?»**
2. **«Как менялось число рейсов по месяцам?»**
3. **«Связано ли расстояние со временем в пути?»**

Требования к каждому графику: название, подписи осей с единицами измерения,
осмысленный порядок категорий.

In [ ]:
# 1) выручка по перевозчикам
fig, ax = plt.subplots(figsize=(9, 4))
...
plt.show()

In [ ]:
# 2) число рейсов по месяцам
...

In [ ]:
# 3) расстояние и время в пути
...

---
# Часть 3. Оформление

График без подписей — это картинка, по которой нельзя сделать вывод. Чек-лист,
по которому стоит проверять каждый график перед отправкой:

| Что | Как |
|---|---|
| название, отвечающее на вопрос | `ax.set_title("...")` |
| подписи осей **с единицами** | `ax.set_xlabel("Выручка, млн руб.")` |
| легенда, если серий больше одной | `ax.legend()` |
| сетка для чтения значений | `ax.grid(True, alpha=0.3)` |
| ось количества начинается с нуля | `ax.set_ylim(0, ...)` |
| подписи не налезают друг на друга | `fig.tight_layout()`, `barh`, поворот |
| подчёркнуто главное | `ax.annotate(...)`, выделение цветом |

Сравним один и тот же график без оформления и с ним.

In [ ]:
otd_by_month = df.groupby("month")["on_time"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ДО: технически верно, но бесполезно
axes[0].plot(otd_by_month.index, otd_by_month.values)
axes[0].set_title("До оформления")

# ПОСЛЕ: тот же график, с которым можно идти к руководителю
ax = axes[1]
ax.plot(otd_by_month.index, otd_by_month.values, marker="o", linewidth=2, color="#4C72B0")
ax.set_title("Доля рейсов без задержки (OTD) по месяцам")
ax.set_xlabel("Месяц")
ax.set_ylabel("OTD, %")
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
ax.axhline(90, color="#C44E52", linestyle="--", label="цель: 90%")
ax.legend()

# подписи значений прямо на точках
for x, y in zip(otd_by_month.index, otd_by_month.values):
    ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8), textcoords="offset points",
                ha="center", fontsize=9)

fig.tight_layout()
plt.show()

Правый график отвечает на вопрос («держим ли мы цель 90%?»), левый — нет, хотя данные
в них одинаковые. Линия цели — важный приём: она превращает «просто числа»
в оценку «хорошо или плохо».

## Ловушка: обрезанная ось

Самая частая манипуляция в отчётах. Если ось количества начать не с нуля, разница
между значениями визуально увеличивается в разы. Посмотрите на два графика ниже —
данные **абсолютно одинаковые**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, ylim, title in [
    (axes[0], (72, 76), "«OTD обвалился!» — ось от 72"),
    (axes[1], (0, 100), "«OTD стабилен» — ось от нуля"),
]:
    ax.bar(otd_by_month.index, otd_by_month.values, color="#4C72B0")
    ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.set_ylabel("OTD, %")

fig.tight_layout()
plt.show()

print("а вот фактические значения:")
print(otd_by_month.round(1).to_string())
print("разница между лучшим и худшим месяцем:",
      round(otd_by_month.max() - otd_by_month.min(), 1), "процентных пункта")

Разница между месяцами — меньше трёх процентных пунктов, но на левом графике она
выглядит как катастрофа. Это не ошибка отрисовки, а способ обмануть читателя,
и встречается он постоянно — в новостях, презентациях, рекламе.

**Правило:** для столбчатых диаграмм ось количества **всегда** начинается с нуля.
Для линейных графиков допустимо обрезать ось, если изменения действительно малы
по сравнению с уровнем, — но тогда это нужно явно оговорить в подписи.

### Задание 2

Возьмите график «выручка по месяцам» и оформите его по чек-листу:

1. название, отвечающее на вопрос (не «График выручки», а, например,
   «Выручка растёт четвёртый месяц подряд»);
2. подписи осей с единицами измерения;
3. ось от нуля;
4. сетка;
5. значения подписаны на точках;
6. **аннотация** на мае: подпишите стрелкой, что рост к апрелю составил всего 2%.

Подсказка для аннотации со стрелкой:

```python
ax.annotate("рост всего 2%",
            xy=(3, 11.3), xytext=(1.5, 9),
            arrowprops=dict(arrowstyle="->", color="black"))
```

In [ ]:
monthly_rev = df.groupby("month")["cost_rub"].sum() / 1_000_000

fig, ax = plt.subplots(figsize=(10, 5))
...
plt.show()

---
# Часть 4. seaborn

seaborn — надстройка над matplotlib. Главное отличие: он работает **прямо с `DataFrame`**,
сам считает агрегаты и сам делает легенду.

```python
sns.barplot(data=df, x="cargo_type", y="cost_rub")   # сам посчитает среднее по группам
```

Ключевой параметр — **`hue`**: разбивка по дополнительному признаку цветом.
Через matplotlib это заняло бы цикл на десять строк.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Средняя стоимость рейса по типу груза (seaborn сам посчитает среднее)
sns.barplot(data=df, y="cargo_type", x="cost_rub", ax=axes[0],
            errorbar=None, color="#4C72B0",
            order=df.groupby("cargo_type")["cost_rub"].mean().sort_values().index)
axes[0].set_title("Средняя стоимость рейса по типу груза")
axes[0].set_xlabel("Средняя стоимость, руб.")
axes[0].set_ylabel("")

# Разбивка цветом через hue: OTD по месяцам и перевозчикам
otd = (df.groupby(["month", "carrier"])["on_time"].mean() * 100).reset_index()
sns.lineplot(data=otd, x="month", y="on_time", hue="carrier", marker="o", ax=axes[1])
axes[1].set_title("OTD по перевозчикам и месяцам")
axes[1].set_xlabel("Месяц")
axes[1].set_ylabel("OTD, %")
axes[1].legend(title="Перевозчик", fontsize=8)

fig.tight_layout()
plt.show()

## «Ящик с усами» (boxplot) — как его читать

Самый информативный график для сравнения групп, но его нужно уметь читать:

- **линия внутри ящика** — медиана;
- **сам ящик** — от Q1 до Q3, то есть центральные 50% данных (это IQR из темы 10);
- **усы** — размах без выбросов (обычно до 1.5 IQR);
- **точки за усами** — выбросы.

Одним графиком показаны сразу центр, разброс и выбросы по каждой группе.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

order = df.groupby("cargo_type")["cost_rub"].median().sort_values().index
sns.boxplot(data=df, x="cargo_type", y="cost_rub", ax=ax, order=order, color="#4C72B0")

ax.set_title("Распределение стоимости рейса по типам груза")
ax.set_xlabel("")
ax.set_ylabel("Стоимость рейса, руб.")
ax.tick_params(axis="x", rotation=20)

fig.tight_layout()
plt.show()

In [ ]:
# Теплокарта: выручка на пересечении маршрута и типа груза
pivot = df.pivot_table(index="route", columns="cargo_type",
                       values="cost_rub", aggfunc="sum", fill_value=0) / 1_000_000

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues", ax=ax,
            cbar_kws={"label": "Выручка, млн руб."})
ax.set_title("Выручка по маршрутам и типам груза, млн руб.")
ax.set_xlabel("")
ax.set_ylabel("")

fig.tight_layout()
plt.show()

### Задание 3

1. Постройте `boxplot` стоимости рейса **по перевозчикам**. У кого разброс больше всех?
   У кого медиана выше?
2. Постройте `histplot` (seaborn) распределения веса груза. Сравните с распределением
   стоимости из части 2: у какого показателя перекос заметнее?
3. Постройте `scatterplot` (seaborn): расстояние по оси X, стоимость по оси Y,
   цвет — тип груза (`hue="cargo_type"`). Что видно на этом графике?
4. Постройте теплокарту: средняя задержка по маршрутам и перевозчикам.
   Какое пересечение самое проблемное?

In [ ]:
# 1)
...

In [ ]:
# 2) и 3)
...

In [ ]:
# 4)
...

---
# Часть 5. Plotly: интерактивные графики

matplotlib делает картинку. Plotly делает **интерактивный** график: при наведении
показывает значения, позволяет приближать и отключать серии в легенде.

Где это нужно: когда отчёт смотрят другие люди и у них возникают вопросы «а сколько
именно вот здесь». Где не нужно: в печатном документе и в презентации — там картинка.

Самый простой способ — `plotly.express`, он работает как seaborn: получает `DataFrame`
и сам всё строит.

In [ ]:
import plotly.express as px

monthly = (df.groupby("month")
           .agg(revenue=("cost_rub", "sum"), trips=("trip_id", "count"))
           .reset_index())
monthly["revenue_mln"] = (monthly["revenue"] / 1_000_000).round(2)

fig = px.line(monthly, x="month", y="revenue_mln", markers=True,
              title="Выручка по месяцам (наведите курсор на точку)",
              labels={"month": "Месяц", "revenue_mln": "Выручка, млн руб."})
fig.update_yaxes(range=[0, monthly["revenue_mln"].max() * 1.15])
fig.show()

In [ ]:
# Точечный график: при наведении покажем номер рейса и маршрут
sample = df.sample(300, random_state=1)

fig = px.scatter(sample, x="weight_t", y="cost_rub", color="cargo_type",
                 hover_data=["trip_id", "route", "distance_km"],
                 title="Вес и стоимость рейса по типам груза",
                 labels={"weight_t": "Вес, т", "cost_rub": "Стоимость, руб.",
                         "cargo_type": "Тип груза"})
fig.show()

Наведите курсор на любую точку — Plotly покажет номер рейса, маршрут и расстояние.
Эти данные не нарисованы на графике, но доступны по запросу: в этом и смысл интерактива.

## Сохранение в файл

Интерактивный график сохраняют в `.html` — его можно открыть в любом браузере
и отправить коллеге.

> Параметр `include_plotlyjs=True` встраивает библиотеку внутрь файла: он станет весить
> около 5 МБ, зато **откроется без интернета**. Если ставить `"cdn"`, файл будет
> крошечным, но без сети покажет пустую страницу. Для учебной работы берите `True`.

In [ ]:
from pathlib import Path

Path("charts").mkdir(exist_ok=True)
fig.write_html("charts/trips_scatter.html", include_plotlyjs=True)

size_mb = Path("charts/trips_scatter.html").stat().st_size / 1024 / 1024
print(f"сохранено: charts/trips_scatter.html ({size_mb:.1f} МБ)")
print("откройте файл двойным щелчком — он работает без интернета")

### Задание 4

1. Постройте интерактивную столбчатую диаграмму (`px.bar`) выручки по маршрутам.
   Отсортируйте столбцы по значению.
2. Добавьте в `hover_data` число рейсов по маршруту, чтобы при наведении было видно
   и выручку, и количество.
3. Сохраните график в `charts/revenue_by_route.html` и откройте файл в браузере.
4. Постройте `px.box` стоимости рейса по типам груза. Чем интерактивный «ящик»
   удобнее статичного?

In [ ]:
# 1) - 3)
route_stats = (df.groupby("route")
               .agg(revenue=("cost_rub", "sum"), trips=("trip_id", "count"))
               .reset_index()
               .sort_values("revenue"))

fig = ...
...

In [ ]:
# 4)
...

---
# Часть 6. Мини-проект: дашборд

**Дашборд** — несколько графиков на одном листе, вместе отвечающих на вопрос
«как идут дела». Главное требование: каждый график отвечает на свой вопрос,
и вместе они не повторяют друг друга.

### Задание 5

Соберите дашборд 2×2 по нашим данным. Четыре графика:

| Позиция | График | Вопрос |
|---|---|---|
| верх слева | динамика выручки по месяцам | растём или падаем? |
| верх справа | выручка по маршрутам (`barh`, сортировка) | на чём зарабатываем? |
| низ слева | распределение стоимости рейса (гистограмма) | какие рейсы типичны? |
| низ справа | OTD по перевозчикам (`barh`) с линией цели 90% | кто работает качественно? |

Требования:

- у каждого графика название и подписи осей с единицами;
- оси количества от нуля;
- общий заголовок через `fig.suptitle(...)`;
- `fig.tight_layout()`;
- сохранить в `charts/dashboard.png` через `fig.savefig(..., dpi=150, bbox_inches="tight")`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# верх слева: динамика выручки


# верх справа: выручка по маршрутам


# низ слева: распределение стоимости


# низ справа: OTD по перевозчикам


fig.suptitle("Перевозки: февраль-май 2025", fontsize=15)
fig.tight_layout()
...
plt.show()

### Задание 6. Выводы по дашборду

Напишите четыре вывода — по одному на каждый график. В каждом должно быть **число**
и то, что оно означает.

Затем ответьте: какой **пятый** график вы бы добавили на этот дашборд и на какой вопрос
он отвечал бы? Почему именно он, а не ещё одна разбивка выручки?

> **Выводы:**
>
> 1. (динамика)
> 2. (маршруты)
> 3. (распределение)
> 4. (OTD)
>
> **Пятый график:** ... потому что ...

---
## Шпаргалка

```python
# каркас
fig, ax = plt.subplots(figsize=(10, 5))
fig, axes = plt.subplots(2, 2, figsize=(14, 9))      # сетка

# типы графиков
ax.bar(x, y);  ax.barh(y, x)        # сравнение категорий
ax.plot(x, y, marker="o")           # динамика
ax.hist(values, bins=30)            # распределение
ax.scatter(x, y, alpha=0.3)         # связь двух величин

# оформление
ax.set_title("...");  ax.set_xlabel("... , ед.");  ax.set_ylabel("...")
ax.set_ylim(0, None)                # ось от нуля
ax.grid(True, alpha=0.3)
ax.legend()
ax.axhline(90, linestyle="--")      # линия цели
ax.annotate("текст", xy=(x, y), xytext=(x2, y2),
            arrowprops=dict(arrowstyle="->"))
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig("charts/plot.png", dpi=150, bbox_inches="tight")

# seaborn
sns.set_theme(style="whitegrid")
sns.barplot(data=df, x="col", y="val", hue="group", errorbar=None)
sns.boxplot(data=df, x="group", y="val")
sns.histplot(data=df, x="val", bins=30)
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues")

# plotly
import plotly.express as px
fig = px.bar(data, x="x", y="y", hover_data=["col"], title="...")
fig.show()
fig.write_html("charts/plot.html", include_plotlyjs=True)
```

## Отправляем работу

```
# терминал
git add .
git commit -m "Практика 11: визуализация данных, дашборд"
git push
```

> В `.gitignore` стоит добавить `charts/*.html` — html-файлы Plotly весят по 5 МБ
> и в репозитории не нужны. А вот `charts/dashboard.png` закоммитьте: это результат работы.

## Чек-лист сдачи

- [ ] Задание 1: три графика под три вопроса, тип выбран осознанно
- [ ] Задание 2: график оформлен по чек-листу, есть аннотация про рост в 2%
- [ ] Задание 3: boxplot, histplot, scatterplot с `hue`, теплокарта
- [ ] Задание 4: интерактивный график с `hover_data`, сохранён в html
- [ ] Задание 5: дашборд 2×2 сохранён в `charts/dashboard.png`
- [ ] Задание 6: четыре вывода с числами + предложение пятого графика
- [ ] У всех графиков есть названия и подписи осей с единицами измерения
- [ ] Всё запушено в репозиторий

## Что унести с пары

Технически график строится одной строкой. Всё остальное время уходит на решения,
которые и отличают рабочий график от картинки:

1. **Тип графика определяется вопросом.** Сравнение категорий — столбцы, динамика —
   линия, распределение — гистограмма, связь — точки.
2. **Ось количества начинается с нуля.** Иначе разница в три процентных пункта выглядит
   как обвал — и вы обманули читателя, даже если не хотели.
3. **Подписи обязательны.** График без единиц измерения на осях не доказывает ничего.
4. **Сортируйте категории.** Это бесплатно и сразу делает график понятнее.

И последнее: прежде чем рисовать, сформулируйте одним предложением, что именно должен
увидеть читатель. Если сформулировать не получается — график не нужен.